# Дообучение реранкера на разговорных вопросах к корпусу законов

На 100 независимых вопросах обычным языком нужная статья часто есть среди кандидатов поиска, но исходный реранкер
`BAAI/bge-reranker-v2-m3` ставит ей низкую оценку: пары «разговорный вопрос — текст закона» он почти не видел.
Ноутбук дообучает его на вопросах, которые Claude написал по текстам статей корпуса
(`prepare_reranker_colloquial.py`: 1–2 вопроса на статью, проверенная дословная цитата, отрицательные примеры —
чанки других статей из выдачи нынешнего поиска).

**Лучшая точка обучения выбирается по настоящим разговорным вопросам** (dev: 50 + 59 вопросов, их писали не для
обучения), а не по сгенерированным. 100 независимых вопросов сюда не попадают — на них итоговая проверка дома.

### Что сделать
1. Загрузить `reranker_colloquial_data.zip` (из `data/laws/reranker_colloquial/colab/`) на Google Drive
   в `MyDrive/rag/`.
2. **Runtime → Change runtime type → T4 GPU** (или лучше).
3. **Runtime → Run all**. Обучение ~40–60 минут на T4.
4. Скачать с Drive `MyDrive/rag/models/reranker-bge-m3-laws-colloquial.zip`, распаковать в папку проекта `models/`
   (получится `models/reranker-bge-m3-laws-colloquial/`), а `reranker_colloquial_results.json` — в
   `data/laws/reranker_colloquial/`.

In [ ]:
# --- Окружение: Google Drive, пакеты, данные ---
import os
import sys
import shutil
import subprocess
import zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
DRIVE_DIR = Path("/content/drive/MyDrive/rag")
WORK = Path("/content") if IN_COLAB else Path.cwd()


def sh(cmd):
    # Вывод команды — в ячейку (иначе в Jupyter он не виден)
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)


if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Версии как в requirements.txt проекта; torch / transformers — колабовские
    sh('pip install -q "sentence-transformers==6.1.0" "datasets==5.1.0" "accelerate==1.15.0"')

archive = DRIVE_DIR / "reranker_colloquial_data.zip"
DATA = WORK / "reranker_colloquial"
if not (DATA / "train_pairs.jsonl").exists():
    if not archive.exists():
        raise FileNotFoundError(f"Нет {archive}: загрузите reranker_colloquial_data.zip на Google Drive в MyDrive/rag/")
    local = WORK / archive.name
    shutil.copy(archive, local)  # распаковка прямо с Drive может прочитать файл не полностью
    with zipfile.ZipFile(local) as z:
        assert z.testzip() is None, "Архив повреждён — загрузите его на Drive заново"
        z.extractall(WORK)
print(sorted(p.name for p in DATA.iterdir()))

## 0. Настройки

In [ ]:
import gc
import json
import math
import time

import pandas as pd
import torch

BASE_RERANKER = "BAAI/bge-reranker-v2-m3"
RUN_NAME = "reranker-bge-m3-laws-colloquial"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "Нужна видеокарта: Runtime → Change runtime type → T4 GPU"

# Как в notebooks/finetune_reranker.ipynb (дообучение на Adilet): lr меньше, чем обычно, — bge уже сильный реранкер;
# gradient checkpointing экономит видеопамять T4 ценой ~30% скорости
BATCH_SIZE, GRAD_ACCUM, LR = 16, 2, 1e-5
EPOCHS = 2
WARMUP_RATIO = 0.1
MAX_LEN = 256              # вопрос + чанк 500 символов ≤ ~260 токенов
EVALS_PER_EPOCH = 4        # dev-оценка четыре раза за эпоху — лучшая точка выбирается по ней
SEED = 42
USE_BF16 = torch.cuda.get_device_capability()[0] >= 8   # A100 / L4 — bf16, T4 — fp16

CHECKPOINT_DIR = WORK / "checkpoints" / RUN_NAME
OUTPUT_DIR = WORK / "out" / RUN_NAME
gpu = torch.cuda.get_device_name(0)
print(gpu, "bf16" if USE_BF16 else "fp16")

## 1. Данные

In [ ]:
def read_jsonl(name):
    with open(DATA / name, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


train_pairs = read_jsonl("train_pairs.jsonl")
val_samples = read_jsonl("val_generated_samples.jsonl")
dev_samples = read_jsonl("dev_samples.jsonl")
summary = json.loads((DATA / "summary.json").read_text(encoding="utf-8"))

n_pos = sum(p["label"] for p in train_pairs)
n_neg = len(train_pairs) - n_pos
POS_WEIGHT = n_neg / n_pos
print(f"Обучающих пар: {len(train_pairs)} (положительных {n_pos}, отрицательных {n_neg}), pos_weight {POS_WEIGHT:.2f}")
print(f"Сгенерированных вопросов для контроля (val): {len(val_samples)}; настоящих разговорных (dev): {len(dev_samples)}")
print("Статистика подготовки:", summary["stats"])
print("Примеры вопросов:")
for p in [p for p in train_pairs if p["label"] == 1][:5]:
    print(" -", p["query"][:150])

## 2. Оценка до обучения

`CrossEncoderRerankingEvaluator` переставляет 20 кандидатов нынешнего поиска (как реранкер в веб-демо) и считает
MRR@10. Вопросы, у которых правильной статьи нет среди кандидатов, он пропускает, поэтому число выше итогового —
оно нужно только для сравнения точек обучения. Итоговые метрики считает дома `eval_laws_search.py`.

In [ ]:
from datasets import Dataset
from sentence_transformers.cross_encoder import CrossEncoder, CrossEncoderTrainer, CrossEncoderTrainingArguments
from sentence_transformers.cross_encoder.evaluation import CrossEncoderRerankingEvaluator
from sentence_transformers.evaluation import SequentialEvaluator
from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss


def reranking_evaluator(samples, name):
    return CrossEncoderRerankingEvaluator(
        samples=[{k: s[k] for k in ("query", "positive", "documents")} for s in samples],
        at_k=10, always_rerank_positives=False, name=name, batch_size=64,
    )


dev_evaluator = reranking_evaluator(dev_samples, "dev")
val_evaluator = reranking_evaluator(val_samples, "val")
evaluator = SequentialEvaluator([dev_evaluator, val_evaluator])


def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()


base = CrossEncoder(BASE_RERANKER, device=DEVICE, max_length=MAX_LEN)
before = {**dev_evaluator(base), **val_evaluator(base)}
print("До обучения:", {k: round(v, 4) for k, v in before.items() if "mrr" in k or "ndcg" in k})
del base
free_gpu()

## 3. Дообучение

`CrossEncoderTrainer` + `BinaryCrossEntropyLoss(pos_weight)`, как в прошлом дообучении. Лучшая точка — по MRR@10
на dev (настоящие разговорные вопросы); в конце загружается она.

In [ ]:
steps_per_epoch = math.ceil(len(train_pairs) / (BATCH_SIZE * GRAD_ACCUM))
eval_steps = max(1, steps_per_epoch // EVALS_PER_EPOCH)
print(f"Шагов на эпоху: {steps_per_epoch}, оценка каждые {eval_steps} шагов")

model = CrossEncoder(BASE_RERANKER, device=DEVICE, max_length=MAX_LEN)
# Включаем у модели напрямую: Trainer из Transformers v5 передаёт аргументы, которых нет
# в CrossEncoder.gradient_checkpointing_enable (sentence-transformers 6.1)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
train_dataset = Dataset.from_dict({k: [p[k] for p in train_pairs] for k in ("query", "passage", "label")}).shuffle(seed=SEED)

args = CrossEncoderTrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=WARMUP_RATIO,          # в Transformers v5 дробное значение означает долю шагов
    fp16=not USE_BF16,
    bf16=USE_BF16,
    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="steps",
    save_steps=eval_steps,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_dev_mrr@10",
    greater_is_better=True,
    logging_steps=50,
    seed=SEED,
    dataloader_num_workers=2,
    report_to="none",
)
trainer = CrossEncoderTrainer(
    model=model, args=args, train_dataset=train_dataset,
    loss=BinaryCrossEntropyLoss(model, pos_weight=torch.tensor(POS_WEIGHT, device=model.device)),
    evaluator=evaluator,
)
t0 = time.time()
trainer.train()
train_minutes = (time.time() - t0) / 60
history = pd.DataFrame(trainer.state.log_history)
print(f"Обучение заняло {train_minutes:.0f} мин")
cols = [c for c in ("epoch", "eval_dev_mrr@10", "eval_val_mrr@10") if c in history]
display(history.dropna(subset=["eval_dev_mrr@10"])[["step", *cols]].round(4))

## 4. До / после и сохранение

In [ ]:
after = {**dev_evaluator(model), **val_evaluator(model)}
table = pd.DataFrame({"до": before, "после": after}).loc[[k for k in before if "mrr" in k or "ndcg" in k]]
display(table.round(4))

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
model.save(str(OUTPUT_DIR))
results = {
    "base": BASE_RERANKER, "model": f"models/{RUN_NAME}", "gpu": gpu, "train_minutes": round(train_minutes, 1),
    "hyperparams": {"batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM, "lr": LR, "epochs": EPOCHS,
                    "warmup_ratio": WARMUP_RATIO, "max_len": MAX_LEN, "evals_per_epoch": EVALS_PER_EPOCH, "seed": SEED,
                    "precision": "bf16" if USE_BF16 else "fp16", "pos_weight": POS_WEIGHT},
    "data": {**summary, "train_pairs_used": len(train_pairs)},
    "selection_metric": "MRR@10 на dev (настоящие разговорные вопросы: 50 + 59), топ-20 кандидатов поиска",
    "before": before, "after": after,
    "history": history.where(history.notna(), None).to_dict(orient="records"),
}
DRIVE_OUT = DRIVE_DIR / "models"
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
zip_base = WORK / "out" / RUN_NAME
shutil.make_archive(str(zip_base), "zip", root_dir=OUTPUT_DIR.parent, base_dir=RUN_NAME)
shutil.copy(f"{zip_base}.zip", DRIVE_OUT / f"{RUN_NAME}.zip")
(DRIVE_DIR / "reranker_colloquial_results.json").write_text(json.dumps(results, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"Модель: {DRIVE_OUT / (RUN_NAME + '.zip')}")
print(f"Отчёт:  {DRIVE_DIR / 'reranker_colloquial_results.json'}")

## Дальше — дома
1. Распаковать `reranker-bge-m3-laws-colloquial.zip` в `models/` проекта, `reranker_colloquial_results.json` —
   в `data/laws/reranker_colloquial/`.
2. Сравнить с исходным реранкером на dev-наборах, затем один раз на 100 независимых вопросах:
   `python eval_laws_search.py --reranker models/reranker-bge-m3-laws-colloquial --questions ...`